# TRACK-FA Entropy and Mutual-Information Feature Selection

This notebook evaluates entropy-guided feature selection for TRACK-FA progression biomarkers.

Pipeline:
1. Load `data/processed/trackfa_pairs_drop3poms.csv`, convert paired baseline/follow-up rows into long visit-level rows, and build baseline/change-score views.
2. Estimate mutual information between candidate features and visit status, baseline mFARS, and mFARS change.
3. Build global and group-wise top-k feature sets from the mutual-information rankings.
4. Evaluate LDA visit-separation models and regression models with full and entropy-selected feature sets.
5. Display feature rankings, selected feature registries, single-feature sensitivity, and model performance tables.

Model overview:
LDA receives visit-level TRACK-FA feature vectors and outputs a one-dimensional visit-separation score. ElasticNet, Ridge, and PLS receive baseline feature vectors and output clinical targets such as baseline mFARS or change in mFARS. These outputs are evaluated with regression metrics and paired progression effect sizes.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project imports and data paths from either the repo root or notebooks folder.
def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")


_REPO_ROOT = find_project_root(Path.cwd())
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from src.config import set_global_seeds  # noqa: E402
from src.data.qc import standardize_train_test  # noqa: E402
from src.eval.cv import lda_loocv, tune_and_run_regression_loocv  # noqa: E402
from src.eval.metrics import (  # noqa: E402
    bootstrap_ci_d,
    compute_cohens_d,
    compute_srm,
    paired_deltas_from_long,
)
from src.features.entropy import (  # noqa: E402
    mi_feature_vs_binary_label,
    mi_feature_vs_continuous_target,
    rank_features_by_mi,
)
from src.features.selection import (  # noqa: E402
    _global_rank,
    make_selection_fn,
    select_topk_global,
    select_topk_by_group,
)


## Load dependencies


In [2]:
set_global_seeds(42)

from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long

RANDOM_SEED = 42
N_BOOT = 2000  # Number of bootstrap resamples for confidence intervals.
K_VALUES = [1, 2, 3, 5]

DATA_PATH = _REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
# Load TRACK-FA paired rows, infer feature groups, and convert to visit-level rows.
pairs = pd.read_csv(DATA_PATH)
groups = infer_trackfa_feature_groups(pairs)
df_long = trackfa_pairs_to_long(pairs)
subject_col = "pair_id"

background = list(groups.background)
structural = list(groups.poms + groups.brainspinemorph)
structural_ext = list(structural)
diffusion = list(groups.braindti)

FEATURE_SETS = {
    "background": background,
    "structural": structural,
    "structural_ext": structural_ext,
    "diffusion": diffusion,
    "background_structural": background + structural,
    "background_structural_ext": background + structural_ext,
    "background_diffusion": background + diffusion,
    "structural_diffusion": structural + diffusion,
    "structural_ext_diffusion": structural_ext + diffusion,
    "background_structural_diffusion": background + structural + diffusion,
    "background_structural_ext_diffusion": background + structural_ext + diffusion,
}

ALL_FEATURES = sorted(set(background + structural_ext + diffusion))

# Reporting groups for mutual-information tables.
FEATURE_GROUPS = {
    "background": background,
    "structural": structural,
    "structural_ext": structural_ext,
    "diffusion": diffusion,
}

print("Pairs shape:", pairs.shape)
print("Long shape:", df_long.shape)
print("Pair column:", subject_col, "| n_pairs:", df_long[subject_col].nunique())
print("Feature counts:", {k: len(v) for k, v in FEATURE_GROUPS.items()})

required_targets = ["FARS", "SARA"]
missing = [c for c in required_targets if c not in df_long.columns]
if missing:
    raise KeyError(f"Missing required target columns after TRACK-FA long conversion: {missing}")

paired_subjects = df_long.groupby(subject_col)["visit"].nunique()
paired_subjects = paired_subjects[paired_subjects == 2].index

# Baseline rows provide regression inputs for clinical score prediction.
df_baseline = df_long[df_long["visit"] == 1].copy()
df_baseline = df_baseline.rename(columns={"FARS": "FARS1", "SARA": "SARA1"})

# Delta rows provide follow-up-minus-baseline outcomes and feature changes.
delta_rows = []
for pid, g in df_long[df_long[subject_col].isin(paired_subjects)].groupby(subject_col):
    g = g.sort_values("visit")
    if set(g["visit"]) != {1, 2}:
        continue
    v1 = g[g["visit"] == 1].iloc[0]
    v2 = g[g["visit"] == 2].iloc[0]
    row = {subject_col: pid, "dFARS": v2["FARS"] - v1["FARS"], "dSARA": v2["SARA"] - v1["SARA"]}
    for f in ALL_FEATURES:
        if f in df_long.columns:
            row[f] = v2[f] - v1[f]
    delta_rows.append(row)
df_delta = pd.DataFrame(delta_rows)

print("Paired rows for progression:", len(paired_subjects))
print("Baseline rows:", df_baseline.shape, "| Delta rows:", df_delta.shape)


Pairs shape: (207, 455)
Long shape: (414, 163)
Pair column: pair_id | n_pairs: 207
Feature counts: {'background': 6, 'structural': 42, 'structural_ext': 42, 'diffusion': 108}
Paired rows for progression: 207
Baseline rows: (207, 163) | Delta rows: (207, 155)


## Score single-feature progression


In [3]:
feature_memberships = {}
for g, feats in FEATURE_GROUPS.items():
    for f in feats:
        feature_memberships.setdefault(f, set()).add(g)

features_for_mi = ALL_FEATURES

mi_visit_rows = []
visit_y = (df_long["visit"].values == 2).astype(int)
for f in features_for_mi:
    if f not in df_long.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_binary_label(df_long[f], visit_y, is_discrete=(f == "sex"))
    mi_visit_rows.append({"feature": f, "mi_visit": mi, "n_visit": n})
mi_visit_df = pd.DataFrame(mi_visit_rows)

mi_fars_rows = []
for f in features_for_mi:
    if f not in df_baseline.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_continuous_target(df_baseline[f], df_baseline["FARS1"], is_discrete_x=(f == "sex"))
    mi_fars_rows.append({"feature": f, "mi_fars1": mi, "n_fars1": n})
mi_fars_df = pd.DataFrame(mi_fars_rows)

mi_dfars_rows = []
for f in features_for_mi:
    if f not in df_delta.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_continuous_target(df_delta[f], df_delta["dFARS"], is_discrete_x=(f == "sex"))
    mi_dfars_rows.append({"feature": f, "mi_dfars": mi, "n_dfars": n})
mi_dfars_df = pd.DataFrame(mi_dfars_rows)

mi_df = mi_visit_df.merge(mi_fars_df, on="feature", how="outer").merge(mi_dfars_df, on="feature", how="outer")

rows_e = []
for _, r in mi_df.iterrows():
    f = r["feature"]
    groups = sorted(feature_memberships.get(f, []))
    if not groups:
        continue
    for g in groups:
        rows_e.append({
            "feature": f, "group": g,
            "mi_visit": r.get("mi_visit"), "n_visit": r.get("n_visit"),
            "mi_fars1": r.get("mi_fars1"), "n_fars1": r.get("n_fars1"),
            "mi_dfars": r.get("mi_dfars"), "n_dfars": r.get("n_dfars"),
        })
single_feature_entropy_df = pd.DataFrame(rows_e)
display(single_feature_entropy_df)


,feature,group,mi_visit,n_visit,mi_fars1,n_fars1,mi_dfars,n_dfars
0,AD_ACR,diffusion,0.004595,414,0.237743,207,0.209396,207
1,AD_ALIC,diffusion,0.011839,414,0.277925,207,0.275481,207
2,AD_CP,diffusion,0.006530,414,0.244688,207,0.241357,207
3,AD_CST,diffusion,0.002122,414,0.240053,207,0.197513,207
4,AD_Cing,diffusion,0.005006,414,0.304744,207,0.262334,207
...,...,...,...,...,...,...,...,...
193,sex,background,0.000000,414,0.042882,207,0.000000,207
194,x3rd_Ventricle,structural,0.002924,414,0.328336,207,0.242578,207
195,x3rd_Ventricle,structural_ext,0.002924,414,0.328336,207,0.242578,207
196,x4th_Ventricle,structural,0.002786,414,0.228992,207,0.181301,207


## Estimate mutual information


In [4]:
single_rows = []
paired = df_long[df_long[subject_col].isin(paired_subjects)].copy()
for f in ALL_FEATURES:
    if f not in paired.columns:
        continue
    tmp = paired[[subject_col, "visit", f]].dropna().copy()
    if tmp.empty:
        continue
    deltas = paired_deltas_from_long(tmp.rename(columns={f: "value"}), subject_col, "visit", "value")
    d_out = compute_cohens_d(deltas)
    srm_out = compute_srm(deltas)
    tmp_oof = tmp.rename(columns={f: "value"})
    _, d_lo, d_hi = bootstrap_ci_d(tmp_oof, subject_col, "visit", "value", n_boot=N_BOOT, seed=RANDOM_SEED)
    if f in background:
        g = "background"
    elif f in structural:
        g = "structural"
    elif f in structural_ext:
        g = "structural_ext"
    elif f in diffusion:
        g = "diffusion"
    else:
        g = "unknown"
    single_rows.append({
        "feature": f, "group": g,
        "d_feature": d_out["d"], "srm_feature": srm_out["srm"],
        "d_ci_low": d_lo, "d_ci_high": d_hi,
        "n_subjects": d_out["n"],
        "mean_diff": d_out["mean"], "sd_diff": d_out["sd"],
    })
single_feature_d_df = pd.DataFrame(single_rows).sort_values("d_feature", ascending=False)
display(single_feature_d_df)


,feature,group,d_feature,srm_feature,d_ci_low,d_ci_high,n_subjects,mean_diff,sd_diff
150,x3rd_Ventricle,structural,0.389207,0.389207,0.278132,0.519447,207,21.910314,56.294723
151,x4th_Ventricle,structural,0.385422,0.385422,0.253905,0.525049,207,35.902556,93.151180
66,Lateral_Ventricle,structural,0.385271,0.385271,0.289335,0.528855,207,382.529111,992.882623
116,RD_SCP,structural,0.260127,0.260127,0.122014,0.393669,207,0.000011,0.000041
84,MD_SCP,structural,0.233416,0.233416,0.094469,0.368001,207,0.000008,0.000036
...,...,...,...,...,...,...,...,...,...
139,disease_duration,background,NaN,NaN,NaN,NaN,207,0.000000,0.000000
141,gaa_1,background,NaN,NaN,NaN,NaN,207,0.000000,0.000000
142,gaa_2,background,NaN,NaN,NaN,NaN,202,0.000000,0.000000
143,onset_age,background,NaN,NaN,NaN,NaN,207,0.000000,0.000000


## Register entropy-selected feature sets


In [5]:
registry_rows = []
for task, source in [("lda_visit", "mi_visit"), ("reg_fars1", "mi_fars1"), ("reg_dfars", "mi_dfars")]:
    global_rank = _global_rank(single_feature_entropy_df, source)
    for k in K_VALUES:
        k2 = min(int(k), len(global_rank))
        registry_rows.append({
            "task": task,
            "selection_mode": "entropy_topk_global",
            "entropy_source": source,
            "k_selected": k2,
            "selected_features": str(global_rank[:k2]),
        })
    group_to_rank = {g: _global_rank(single_feature_entropy_df, source, group=g) for g in FEATURE_GROUPS.keys()}
    for k in K_VALUES:
        selected = []
        for g, rank in group_to_rank.items():
            selected.extend(rank[: min(int(k), len(rank))])
        seen = set()
        sel = []
        for f in selected:
            if f not in seen:
                seen.add(f)
                sel.append(f)
        registry_rows.append({
            "task": task,
            "selection_mode": "entropy_topk_group",
            "entropy_source": source,
            "k_selected": int(k),
            "selected_features": str(sel),
        })
entropy_selected_feature_sets_df = pd.DataFrame(registry_rows)
display(entropy_selected_feature_sets_df)


,task,selection_mode,entropy_source,k_selected,selected_features
0,lda_visit,entropy_topk_global,mi_visit,1,['sMD_c3c5']
1,lda_visit,entropy_topk_global,mi_visit,2,"['sMD_c3c5', 'AD_ALIC']"
2,lda_visit,entropy_topk_global,mi_visit,3,"['sMD_c3c5', 'AD_ALIC', 'MD_SLF']"
3,lda_visit,entropy_topk_global,mi_visit,5,"['sMD_c3c5', 'AD_ALIC', 'MD_SLF', 'MD_mLEM', '..."
4,lda_visit,entropy_topk_group,mi_visit,1,"['age', 'sMD_c3c5', 'AD_ALIC']"
5,lda_visit,entropy_topk_group,mi_visit,2,"['age', 'disease_duration', 'sMD_c3c5', 'sAD_c..."
6,lda_visit,entropy_topk_group,mi_visit,3,"['age', 'disease_duration', 'gaa_1', 'sMD_c3c5..."
7,lda_visit,entropy_topk_group,mi_visit,5,"['age', 'disease_duration', 'gaa_1', 'gaa_2', ..."
8,reg_fars1,entropy_topk_global,mi_fars1,1,['FA_SCP']
9,reg_fars1,entropy_topk_global,mi_fars1,2,"['FA_SCP', 'disease_duration']"


## Train entropy-selected models


In [ ]:
results = []


def _make_sel(task, mode, k, src):
    return make_selection_fn(
        task=task, selection_mode=mode, k_selected=k,
        entropy_source=src,
        all_features=ALL_FEATURES,
        feature_groups=FEATURE_GROUPS,
    )


# LDA visit-separation models using complete feature sets.
for fs_name, feats in FEATURE_SETS.items():
    res = lda_loocv(df_long, feats, subject_col=subject_col, visit_col="visit", selection_fn=None)
    best_single_d = float(single_feature_d_df["d_feature"].max()) if len(single_feature_d_df) else np.nan
    results.append({
        "method": "LDA", "task": "separation", "target": "visit_axis",
        "feature_set": fs_name, "n_features": len(feats),
        "selection_mode": "full", "k_selected": np.nan, "entropy_source": np.nan,
        "d_score": res["d_score"], "srm": res["srm"],
        "d_ci_low": res["d_ci_low"], "d_ci_high": res["d_ci_high"],
        "rmse": np.nan, "r2": np.nan,
        "n_subjects": res["n_subjects"],
        "beats_best_single": (
            bool(res["d_score"] > best_single_d)
            if np.isfinite(res["d_score"]) and np.isfinite(best_single_d) else np.nan
        ),
        "notes": "",
    })

# LDA visit-separation models using entropy-selected features.
for selection_mode in ["entropy_topk_group", "entropy_topk_global"]:
    for k in K_VALUES:
        sel_fn = _make_sel("lda_visit", selection_mode, k, "mi_visit")
        res = lda_loocv(df_long, ALL_FEATURES, subject_col=subject_col, visit_col="visit", selection_fn=sel_fn)
        best_single_d = float(single_feature_d_df["d_feature"].max()) if len(single_feature_d_df) else np.nan
        results.append({
            "method": "LDA", "task": "separation", "target": "visit_axis",
            "feature_set": "global_pool", "n_features": np.nan,
            "selection_mode": selection_mode, "k_selected": int(k), "entropy_source": "mi_visit",
            "d_score": res["d_score"], "srm": res["srm"],
            "d_ci_low": res["d_ci_low"], "d_ci_high": res["d_ci_high"],
            "rmse": np.nan, "r2": np.nan,
            "n_subjects": res["n_subjects"],
            "beats_best_single": (
                bool(res["d_score"] > best_single_d)
                if np.isfinite(res["d_score"]) and np.isfinite(best_single_d) else np.nan
            ),
            "notes": "entropy-selected (fold-specific)",
        })

# Single-feature reference rows for progression sensitivity.
if len(single_feature_d_df):
    best_by_d = single_feature_d_df.sort_values("d_feature", ascending=False).iloc[0]["feature"]
    best_by_mi_visit = single_feature_entropy_df.sort_values("mi_visit", ascending=False).iloc[0]["feature"]
    for feat, note in [(best_by_d, "single_feature_best_d"), (best_by_mi_visit, "single_feature_best_mi_visit")]:
        res = lda_loocv(df_long, [feat], subject_col=subject_col, visit_col="visit", selection_fn=None)
        best_single_d = float(single_feature_d_df["d_feature"].max())
        results.append({
            "method": "LDA", "task": "separation", "target": "visit_axis",
            "feature_set": feat, "n_features": 1,
            "selection_mode": "single_feature", "k_selected": 1, "entropy_source": np.nan,
            "d_score": res["d_score"], "srm": res["srm"],
            "d_ci_low": res["d_ci_low"], "d_ci_high": res["d_ci_high"],
            "rmse": np.nan, "r2": np.nan,
            "n_subjects": res["n_subjects"],
            "beats_best_single": (
                bool(res["d_score"] >= best_single_d)
                if np.isfinite(res["d_score"]) else np.nan
            ),
            "notes": note,
        })


def run_regression_suite(df_task, target_col, task_name, entropy_source):
    for fs_name, feats in FEATURE_SETS.items():
        for model_kind, method in [("elasticnet", "ElasticNet"), ("ridge", "Ridge"), ("pls", "PLS")]:
            res = tune_and_run_regression_loocv(
                df_task, feats, target_col=target_col, subject_col=subject_col,
                model_kind=model_kind, selection_fn=None,
            )
            results.append({
                "method": method, "task": "regression", "target": target_col,
                "feature_set": fs_name, "n_features": len(feats),
                "selection_mode": "full", "k_selected": np.nan, "entropy_source": np.nan,
                "d_score": np.nan, "srm": np.nan, "d_ci_low": np.nan, "d_ci_high": np.nan,
                "rmse": res["rmse"], "r2": res["r2"],
                "n_subjects": res["n_subjects"],
                "beats_best_single": np.nan, "notes": "",
            })
    for selection_mode in ["entropy_topk_group", "entropy_topk_global"]:
        for k in K_VALUES:
            sel_fn = _make_sel(task_name, selection_mode, k, entropy_source)
            for model_kind, method in [("elasticnet", "ElasticNet"), ("ridge", "Ridge"), ("pls", "PLS")]:
                res = tune_and_run_regression_loocv(
                    df_task, ALL_FEATURES, target_col=target_col, subject_col=subject_col,
                    model_kind=model_kind, selection_fn=sel_fn,
                )
                results.append({
                    "method": method, "task": "regression", "target": target_col,
                    "feature_set": "global_pool", "n_features": np.nan,
                    "selection_mode": selection_mode, "k_selected": int(k), "entropy_source": entropy_source,
                    "d_score": np.nan, "srm": np.nan, "d_ci_low": np.nan, "d_ci_high": np.nan,
                    "rmse": res["rmse"], "r2": res["r2"],
                    "n_subjects": res["n_subjects"],
                    "beats_best_single": np.nan,
                    "notes": "entropy-selected (fold-specific)",
                })


run_regression_suite(df_baseline, target_col="FARS1", task_name="reg_fars1", entropy_source="mi_fars1")
run_regression_suite(df_delta, target_col="dFARS", task_name="reg_dfars", entropy_source="mi_dfars")

results_df = pd.DataFrame(results)
print("Total result rows:", len(results_df))

entropy_only = results_df[results_df["selection_mode"].isin(["entropy_topk_group", "entropy_topk_global"])].copy()
display(results_df)
display(entropy_only)


## Final output summary


In [ ]:
print("Results rows:", len(results_df))
sep = results_df[results_df["task"] == "separation"].sort_values("d_score", ascending=False)
print("Top separation rows:", len(sep))
reg_fars1 = results_df[(results_df["task"] == "regression") & (results_df["target"] == "FARS1")].sort_values("rmse", ascending=True)
reg_dfars = results_df[(results_df["task"] == "regression") & (results_df["target"] == "dFARS")].sort_values("rmse", ascending=True)
print("Top FARS1 reg rows:", len(reg_fars1), "| dFARS reg rows:", len(reg_dfars))
display(sep.head(10))
display(reg_fars1.head(10))
display(reg_dfars.head(10))
